# Visualización de rutas EVRP (flota de N vehículos)

Carga las salidas de `python -m src.solve` (en `outputs/`) y dibuja:

1. Las rutas sobre un mapa interactivo (folium), **una por vehículo con color propio**: BASE, visitas y cargadores usados.
2. La curva de SOC a lo largo de la ruta de cada vehículo, con la línea del SOC mínimo.

No reejecuta la optimización: solo lee `route_<instancia>.csv` y `metrics_<instancia>.json`.
Ejecuta primero, desde la raíz del proyecto (ejemplo con flota de 2):

```bash
python -m src.solve --instance all --vehicles 2
```

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import folium

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUTPUTS = PROJECT_ROOT / 'outputs'
print('Outputs dir:', OUTPUTS)

In [ ]:
# Paleta de colores por vehículo.
VEHICLE_COLORS = ['#1f77b4', '#d62728', '#2ca02c', '#9467bd', '#ff7f0e', '#17bec7']

# Mosaicos de OpenStreetMap. Los de CARTO ahora exigen API key y, sin ella,
# muestran un aviso de 'API key' sobre el mapa.
TILES_URL = 'https://tile.openstreetmap.org/{z}/{x}/{y}.png'
TILES_ATTR = '&copy; OpenStreetMap contributors'

def load_outputs(instance):
    route = pd.read_csv(OUTPUTS / f'route_{instance}.csv')
    metrics = json.loads((OUTPUTS / f'metrics_{instance}.json').read_text())
    if 'vehicle' not in route.columns:
        route['vehicle'] = 0  # compatibilidad con salidas de 1 vehículo
    return route, metrics

In [ ]:
def route_map(route, metrics, instance):
    center = [route['latitude'].mean(), route['longitude'].mean()]
    # Un solo mapa base con atribucion explicita y limpia.
    m = folium.Map(location=center, zoom_start=12, tiles=None, control_scale=True)
    folium.TileLayer(tiles=TILES_URL, attr=TILES_ATTR, name='mapa', control=False).add_to(m)

    legend_items = []
    for vid, g in route.groupby('vehicle'):
        color = VEHICLE_COLORS[int(vid) % len(VEHICLE_COLORS)]
        coords = list(zip(g['latitude'], g['longitude']))
        folium.PolyLine(coords, color=color, weight=4, opacity=0.85,
                        tooltip=f'Vehiculo {vid}').add_to(m)
        for _, row in g.iterrows():
            t = row['node_type']
            if t == 'base':
                mk_color, icon = 'black', 'home'
            elif t == 'charger':
                mk_color, icon = 'green', 'flash'
            else:
                mk_color, icon = 'blue', 'record'
            popup = f"<b>{row['node_id']}</b> ({t})<br>Vehiculo {vid}<br>SOC: {row['soc_arrival_kwh']} kWh"
            rk = row.get('recharged_kwh', 0)
            try:
                rk = float(rk)
            except (TypeError, ValueError):
                rk = 0.0
            if rk > 0:
                popup += f"<br>Recarga: {rk:.2f} kWh @ {row['recharge_power_kw']} kW"
            folium.CircleMarker(
                [row['latitude'], row['longitude']],
                radius=6 if t == 'visit' else 8,
                color=color if t == 'visit' else mk_color,
                fill=True, fill_color=color if t == 'visit' else mk_color,
                fill_opacity=0.9,
                tooltip=f"{row['node_id']} (V{vid})",
                popup=folium.Popup(popup, max_width=250),
            ).add_to(m)
        legend_items.append((vid, color))

    # Leyenda propia, bien ubicada (abajo a la izquierda), sin encimar atribucion.
    rows = ''.join(
        f'<div><span style="display:inline-block;width:12px;height:12px;'
        f'background:{c};margin-right:6px;"></span>Vehiculo {v}</div>'
        for v, c in legend_items
    )
    legend = f'''
    <div style="position:fixed;bottom:30px;left:12px;z-index:9999;
                background:white;padding:8px 10px;border:1px solid #999;
                border-radius:6px;font-size:12px;box-shadow:0 1px 4px rgba(0,0,0,.3);">
      <b>{instance}</b><br>{rows}
      <div style="margin-top:4px;color:#555;">● base &nbsp; ⚡ cargador</div>
    </div>'''
    m.get_root().html.add_child(folium.Element(legend))
    return m

In [ ]:
def soc_plot(route, metrics, instance):
    cap = metrics['battery_capacity_kwh']
    min_soc = metrics['min_soc_limit_frac'] * cap
    fig, ax = plt.subplots(figsize=(12, 4.5))
    for vid, g in route.groupby('vehicle'):
        color = VEHICLE_COLORS[int(vid) % len(VEHICLE_COLORS)]
        x = range(len(g))
        ax.plot(x, g['soc_arrival_kwh'], marker='o', color=color, label=f'Vehiculo {vid}')
    ax.axhline(min_soc, color='red', ls='--', label=f'SOC minimo ({metrics["min_soc_limit_frac"]*100:.0f}%)')
    ax.axhline(cap, color='green', ls=':', label='Capacidad')
    ax.set_xlabel('Paso en la ruta (por vehiculo)')
    ax.set_ylabel('SOC [kWh]')
    total_e = metrics.get('total_drive_energy_kwh', metrics.get('drive_energy_kwh', 0))
    total_r = metrics.get('total_recharges', metrics.get('n_recharges', 0))
    ax.set_title(f'Perfil de SOC - {instance} (flota {metrics.get("fleet_size", 1)}, '
                 f'energia total {total_e:.1f} kWh, recargas {total_r})')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    return fig

## Instancia: evaluation

In [ ]:
route_eval, metrics_eval = load_outputs('evaluation')
print('Factible:', metrics_eval['feasible'], '| Flota:', metrics_eval.get('fleet_size'),
      '| SOC min %:', round(metrics_eval['min_soc_frac']*100, 1))
soc_plot(route_eval, metrics_eval, 'evaluation')
plt.show()

In [ ]:
route_map(route_eval, metrics_eval, 'evaluation')

## Instancia: competition

In [ ]:
route_comp, metrics_comp = load_outputs('competition')
print('Factible:', metrics_comp['feasible'], '| Flota:', metrics_comp.get('fleet_size'),
      '| SOC min %:', round(metrics_comp['min_soc_frac']*100, 1))
soc_plot(route_comp, metrics_comp, 'competition')
plt.show()

In [ ]:
route_map(route_comp, metrics_comp, 'competition')